In [7]:
# =========================
# 📦 Imports
# =========================
import pandas as pd

# =========================
# 📥 Load datasets
# =========================
flights_df = pd.read_csv("/content/aviationstack_flights_cleaned.csv")
airports_df = pd.read_csv("/content/Airports.csv")

# =========================
# 🔑 STEP 1: Extract used airports from Fact
# =========================
used_airports = pd.unique(
    flights_df[["Departure IATA", "Arrival IATA"]].values.ravel()
)

used_airports = [x for x in used_airports if pd.notna(x)]

print("Total used airports in fact:", len(used_airports))

# =========================
# ✈️ STEP 2: Filter airports dataset
# =========================
airports_filtered = airports_df[
    airports_df["iata_code"].isin(used_airports)
].copy()

print("After filtering:", len(airports_filtered))

# =========================
# 🧹 STEP 3: Keep only useful columns
# =========================
# =========================
# 🧹 STEP 3: Keep only useful columns
# =========================
airports_filtered = airports_filtered[[
    "iata_code",
    "type",
    "name",
    "country_name",
    "continent",
    "elevation_ft",
    "municipality",
    "latitude_deg",
    "longitude_deg"
]]

# =========================
# 🧼 STEP 4: Clean critical missing values
# =========================
# drop rows that can't be used for joins or geo analysis
airports_filtered = airports_filtered.dropna(
    subset=["iata_code", "latitude_deg", "longitude_deg"]
)

# fill optional fields
airports_filtered["name"] = airports_filtered["name"].fillna("Unknown Airport")
airports_filtered["municipality"] = airports_filtered["municipality"].fillna("Unknown")
country_to_continent = {
    "Egypt": "AF",
    "Nigeria": "AF",
    "South Africa": "AF",

    "United States": "NA",
    "Canada": "NA",
    "Mexico": "NA",

    "United Kingdom": "EU",
    "Germany": "EU",
    "France": "EU",
    "Italy": "EU",
    "Spain": "EU",

    "China": "AS",
    "Japan": "AS",
    "India": "AS",
    "United Arab Emirates": "AS",
    "Saudi Arabia": "AS",

    "Brazil": "SA",
    "Argentina": "SA",

    "Australia": "OC",
    "New Zealand": "OC",
}

# fill missing continent using country mapping
airports_filtered["continent"] = airports_filtered.apply(
    lambda row: country_to_continent.get(
        row["country_name"],
        row["continent"]  # keep original if already exists
    ),
    axis=1
)
# =========================
# 🔁 STEP 5: Remove duplicates
# =========================
airports_filtered.drop_duplicates(subset=["iata_code"], inplace=True)

# =========================
# 📊 Final check
# =========================
print("\nFinal Airports Dataset:")
print(airports_filtered.shape)
print(airports_filtered.head())

# =========================
# 💾 SAVE OUTPUT
# =========================
airports_filtered.to_csv("airports_clean_dim.csv", index=False)

print("✅ Clean airports dimension saved!")

Total used airports in fact: 386
After filtering: 386

Final Airports Dataset:
(386, 9)
  iata_code           type                                              name  \
0       LHR  large_airport                           London Heathrow Airport   
1       LAX  large_airport                 Los Angeles International Airport   
2       ORD  large_airport              Chicago O'Hare International Airport   
3       JFK  large_airport             John F. Kennedy International Airport   
4       ATL  large_airport  Hartsfield Jackson Atlanta International Airport   

     country_name continent  elevation_ft municipality  latitude_deg  \
0  United Kingdom        EU          83.0       London     51.470748   
1   United States        NA         125.0  Los Angeles     33.942501   
2   United States        NA         680.0      Chicago     41.978600   
3   United States        NA          13.0     New York     40.639447   
4   United States        NA        1026.0      Atlanta     33.636700   